# Baselines

Two baselines against which QLoRA adapters are compared:
- **B1 -- Zero-shot:** Base model with artist name in prompt, no fine-tuning
- **B2 -- Few-shot:** Base model with artist name + 3 example lyrics in prompt

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers import AutoModelForSequenceClassification, AutoTokenizer as AT
import torch
import pandas as pd
import random

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model_path = "./models/gemma-4-E4B"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
)

clf_path = "./classifier_output/best_model"
clf_model = AutoModelForSequenceClassification.from_pretrained(clf_path)
clf_tokenizer = AT.from_pretrained(clf_path)
clf_model.eval()

labels = clf_model.config.id2label
artists = ["Death", "Gojira", "Meshuggah", "Opeth", "Tool"]
n_samples = 10

In [ ]:
def generate_samples(prompt, n):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    samples = []
    for _ in range(n):
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=512,
                min_new_tokens=200,
                do_sample=True,
                temperature=0.9,
                top_p=0.9,
                repetition_penalty=1.1,
            )
        text = tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True)
        samples.append(text)
    return samples, prompt_len


def classify_samples(samples):
    results = []
    for text in samples:
        enc = clf_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            logits = clf_model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[0]
        results.append({labels[j]: probs[j].item() for j in range(len(labels))})
    return pd.DataFrame(results)

## B1 -- Zero-shot

Artist name in prompt, no examples, no fine-tuning.

In [ ]:
zero_shot = {}

for artist in artists:
    prompt = f"Write song lyrics in the style of {artist}.\n\n"
    samples, prompt_len = generate_samples(prompt, n_samples)
    df = classify_samples(samples)
    zero_shot[artist] = {"samples": samples, "df": df}

    print(f"\n=== {artist} (zero-shot, prompt: {prompt_len} tokens) ===")
    print(f"Mean attribution:\n{df.mean().round(4)}")
    print(f"Std:\n{df.std().round(4)}")

## B2 -- Few-shot (3 examples)

Artist name + 3 training lyrics as in-context examples.

In [ ]:
train_df = pd.read_csv("./data/train.csv")
n_examples = 3
random.seed(42)

few_shot = {}

for artist in artists:
    artist_lyrics = train_df[train_df["artist"] == artist]["lyrics"].tolist()
    examples = random.sample(artist_lyrics, n_examples)

    prompt = f"Write song lyrics in the style of {artist}.\n\n"
    for i, ex in enumerate(examples, 1):
        prompt += f"Example {i}:\n{ex}\n\n"
    prompt += f"Now write new song lyrics in the style of {artist}:\n\n"

    samples, prompt_len = generate_samples(prompt, n_samples)
    df = classify_samples(samples)
    few_shot[artist] = {"samples": samples, "df": df}

    print(f"\n=== {artist} (few-shot, {n_examples} examples, prompt: {prompt_len} tokens) ===")
    print(f"Mean attribution:\n{df.mean().round(4)}")
    print(f"Std:\n{df.std().round(4)}")

## Summary

In [ ]:
print("Target-artist mean attribution (higher = better):\n")
print(f"{'Artist':<12} {'Zero-shot':>10} {'Few-shot':>10}")
print("-" * 34)
for artist in artists:
    zs = zero_shot[artist]["df"][artist].mean()
    fs = few_shot[artist]["df"][artist].mean()
    print(f"{artist:<12} {zs:>10.4f} {fs:>10.4f}")